# Exercise 2.6: Transforming and Merging (Angola IEA and INE trade)

Two sources, two jobs. The survey cleaned in 2.5 becomes analysis ready through
custom functions and `apply`. The INE trade workbooks are then loaded, merged and
stacked.

Every mapping in this notebook is **extracted from a source**, never typed by
hand: variable descriptions come from the SPSS header, country names from the
trade sheet, category names from the workbook that defines them.

**PT:** Duas fontes, dois trabalhos. O inquerito limpo em 2.5 torna-se pronto
para analise com funcoes proprias e `apply`. Depois carregamos, juntamos e
empilhamos os ficheiros de comercio do INE.

Todos os mapeamentos sao **extraidos da fonte**, nunca escritos a mao.

> **Pipeline:** run 2.5 first. Reads `10_cleaned/` and `0_raw/angola`, writes
> `20_processed/`.

### Path Setup (run first)

**PT:** Configuracao dos caminhos.

In [ ]:
import os

import numpy as np
import pandas as pd

DATA_RAW_DIR = '../../data/0_raw/angola'
DATA_CLEAN_DIR = '../../data/10_cleaned'
DATA_PROC_DIR = '../../data/20_processed'

EMPLOYMENT_DIR = 'employment_survey'
TRADE_DIR = 'international_trade'
RAW_FILE = 'IEA_2025_IV_TRIM_IND.sav'

clean_path = os.path.join(DATA_CLEAN_DIR, 'angola_iea_2025q4_clean.csv')
trade_dir = os.path.join(DATA_RAW_DIR, TRADE_DIR)

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
print('Trade workbooks available:')
for name in sorted(os.listdir(trade_dir)):
    print('  ', name)

---

# Part A: the survey

## Task 1: Load the cleaned file and recover its descriptions

CSV keeps no dtypes and no documentation, so both have to be re-established.
The descriptions come back from the SPSS header rather than from a dictionary
typed into this notebook.

**PT:** O CSV nao guarda tipos nem documentacao, por isso ambos tem de ser
restabelecidos. As descricoes voltam do cabecalho SPSS, nao de um dicionario
escrito a mao aqui.

In [ ]:
import pyreadstat

df = pd.read_csv(clean_path, dtype={'nidf': 'string', 'ppno': 'string',
                                    'g_06_id_iea': 'string'})

raw_path = os.path.join(DATA_RAW_DIR, EMPLOYMENT_DIR, RAW_FILE)
_, meta = pyreadstat.read_sav(raw_path, metadataonly=True)
descriptions =   # your code here: lower cased name -> label, only for
               #                  columns present in df

print('Survey:', df.shape)
print('Descriptions recovered:', len(descriptions))
print()
for col in ['atw_pay', 'srh_avn', 'pond_iea_iv_trim_2025_ind']:
    print(f'{col:28s} {descriptions[col][:60]}')

**Questions:**

- How many rows, columns and descriptions did you recover?
- Why rebuild the descriptions from the SPSS file instead of writing them here?
- Read the description of `atw_pay`. Why will Task 3 use it?

**PT:** Quantas linhas, colunas e descricoes recuperou? Porque reconstruir as
descricoes a partir do ficheiro SPSS? Leia a descricao de `atw_pay`.

---

## Task 2: A custom function on one column

`apply` on a Series runs your function once per value. Use it when the rule needs
branching that a vectorised expression cannot express clearly.

The bands are not arbitrary: 15 is the ILO working age threshold and 65 the usual
retirement reference, so the function encodes a definition rather than a
convenience.

**PT:** `apply` numa Serie corre a funcao para cada valor. Use quando a regra tem
ramificacoes. As faixas nao sao arbitrarias: 15 anos e o limiar de idade ativa da
OIT e 65 a referencia de reforma.

In [ ]:
def age_band(age):
    """ILO oriented age band for one person."""
    # your code here: guard the missing value, then band at 15, 25 and 65
    return


df['age_band'] = df['dem_age'].apply(age_band)
df['age_band'].value_counts()

**Questions:**

- How many people fall in each band? What share are children?
- `pd.cut` would be faster. When is a function still the better choice?

**PT:** Quantas pessoas em cada faixa? Que percentagem sao criancas? `pd.cut`
seria mais rapido: quando e que a funcao continua a ser melhor?

---

## Task 3: A custom function across several columns

`apply(axis=1)` passes a whole row, so the function can read many columns at
once. Labour force status is the natural case: it depends on eight answers and no
single-column expression can express it.

The values are Portuguese labels, not codes, because 2.5 loaded the file with
`convert_categoricals=True`. The function reads almost like the questionnaire.

**PT:** `apply(axis=1)` passa a linha inteira, por isso a funcao pode ler varias
colunas. A situacao perante o trabalho depende de oito respostas. Os valores sao
etiquetas em portugues, por isso a funcao le-se quase como o questionario.

In [ ]:
def labour_force_status(row):
    """ILO status for one person, from the survey's own answers.

    Employed: worked for pay or profit, or has a job they were absent from.
    Unemployed: not employed, looked for work, and available to start.
    """
    if row['dem_age'] < 15:
        return 'Outside labour force'

    worked = (row['atw_pay'], row['atw_pft'], row['abs_job'])
    if 'Sim' in worked:
        return 'Employed'

    searched =   # your code here: said Sim to srh_job or srh_bus
    available =   # your code here: said Sim to srh_avn or srh_avl
    if searched and available:
        return 'Unemployed'

    return 'Outside labour force'


df['lf_status'] = df.apply(labour_force_status, axis=1)
df['lf_status'].value_counts()

**Questions:**

- How many are employed, unemployed and outside the labour force?
- Why does availability test `srh_avn` **or** `srh_avl` rather than `srh_avl`
  alone? Check how much of `srh_avl` is missing.
- Would this function be as readable if the file had been loaded with raw codes?
- This is slow. What would you have to give up to vectorise it?

**PT:** Quantos empregados, desempregados e fora da forca de trabalho? Porque a
disponibilidade testa `srh_avn` **ou** `srh_avl`? A funcao seria tao legivel com
codigos numericos? Isto e lento: o que perderia ao vetorizar?

---

## Task 4: Weight the result

Each person represents many Angolans, and `pond_iea_iv_trim_2025_ind` says how
many. An unweighted rate describes the sample; a weighted rate describes the
country.

**PT:** Cada pessoa representa muitos angolanos, e o ponderador diz quantos. Uma
taxa nao ponderada descreve a amostra; uma taxa ponderada descreve o pais.

In [ ]:
def weighted_share(mask, weights):
    """Share of the weighted population selected by `mask`, as a percentage."""
    # your code here: weighted share of the mask, as a percentage
    return


weight = df['pond_iea_iv_trim_2025_ind']
in_labour_force = df['lf_status'].isin(['Employed', 'Unemployed'])

unemployment = weighted_share(df['lf_status'] == 'Unemployed', weight[in_labour_force])
participation = weighted_share(in_labour_force, weight[df['dem_age'] >= 15])

print(f'Unemployment rate:  {unemployment:5.1f}%')
print(f'Participation rate: {participation:5.1f}%')

**Questions:**

- What are the unemployment and participation rates?
- Is this the strict or the relaxed definition, and would INE publish this number?
- Compare the weighted and unweighted rates. Does the difference surprise you?

**PT:** Quais sao as taxas? Esta e a definicao estrita ou alargada? Compare a
taxa ponderada e nao ponderada.

---

## Task 5: Household size without `groupby`

The file has a household size column, but it is empty in every row, so 2.5 never
loaded it. Rebuild it from the roster: count the rows sharing each `nidf`, then
map that count back onto every person.

**PT:** O ficheiro tem uma coluna de dimensao do agregado, mas esta vazia, por
isso 2.5 nao a carregou. Reconstrua contando as linhas por `nidf`.

In [ ]:
df['hh_size'] =   # your code here: map each nidf to how often it appears

per_person = df['hh_size'].mean()
per_household = df.drop_duplicates('nidf')['hh_size'].mean()

print(f'Mean household size per person:    {per_person:.2f}')
print(f'Mean household size per household: {per_household:.2f}')

**Questions:**

- Why do the two means differ? Which answers "the average household has how many
  people"?
- Which one would be wrong to publish as "average household size"?

**PT:** Porque as duas medias diferem? Qual responde a "o agregado medio tem
quantas pessoas"? Qual seria errado publicar?

---

# Part B: the trade workbooks

## Task 6: One loader for seven workbooks

INE publishes seven workbooks with the same human readable layout: two title
rows, the real header, a blank row, a `Total Geral` row, the data, and a source
footer. Writing the cleanup once and calling it repeatedly is the whole reason to
define a function.

Rows are dropped by a **property** (`Descrição` is empty) rather than by
position, so the loader survives INE adding a line next quarter.

**PT:** O INE publica sete ficheiros com o mesmo formato: duas linhas de titulo,
o cabecalho, uma linha vazia, o `Total Geral`, os dados, e o rodape da fonte.
Escrever a limpeza uma vez justifica definir uma funcao. As linhas sao removidas
por uma propriedade, nao por posicao.

In [ ]:
def load_trade_sheet(file_name, sheet_name, label_col):
    """Load one INE trade sheet, stripped of title, total and footer rows.

    `label_col` is the descriptive column, which is empty on exactly the rows we
    do not want.
    """
    path = os.path.join(trade_dir, file_name)
    frame = pd.read_excel(path, sheet_name=sheet_name, skiprows=2)
    frame.columns = frame.columns.str.replace('\n', ' ', regex=False).str.strip()
    frame =   # your code here: keep only rows where label_col is not null
    return frame


PARTNERS = 'Comercio Externo de Bens por Países Parceiros.xlsx'
exports = load_trade_sheet(PARTNERS, 'Exportação por Países (USD)', 'País')
imports = load_trade_sheet(PARTNERS, 'Importação por Países (USD)', 'País')

print('exports:', exports.shape, '| imports:', imports.shape)
exports[['Código', 'País', 'Ano 2025']].head()

**Questions:**

- How many rows does each sheet give? What is the extra one that is not a country?
- Why drop rows by a property rather than by position?
- What unit are the values in? Where does the file say so, and does any column
  name tell you?

**PT:** Quantas linhas tem cada folha? Porque remover linhas por propriedade e
nao por posicao? Em que unidade estao os valores e onde e que o ficheiro o diz?

---

## Task 7: Extract the code to name mapping

The country names live in the sheet. Build the lookup from there rather than
typing 248 names, and it stays correct when INE revises one.

**PT:** Os nomes dos paises estao na propria folha. Construa a correspondencia a
partir dela em vez de escrever 248 nomes a mao.

In [ ]:
country_names =   # your code here: build the code to name lookup from the sheet

print('Countries mapped:', len(country_names))
print('Sample:', {k: country_names[k] for k in list(country_names)[:4]})
print('ZZ means:', country_names['ZZ'])

**Questions:**

- How many countries did you map?
- What is `ZZ`, and should you drop it?

**PT:** Quantos paises mapeou? O que e `ZZ` e deve remove-lo?

---

## Task 8: Merge exports against imports, then classify with `apply(axis=1)`

Both tables carry one row per country, so this is a one to one merge and
`validate` should say so.

The classification then needs both columns at once, which is the second natural
use of `apply(axis=1)`.

**PT:** As duas tabelas tem uma linha por pais, por isso a juncao e um para um.
A classificacao precisa das duas colunas ao mesmo tempo, o segundo uso natural de
`apply(axis=1)`.

In [ ]:
YEAR = 'Ano 2025'

trade = pd.merge(
    exports[['Código', 'País', YEAR]].rename(columns={YEAR: 'exports_thousand_usd'}),
    imports[['Código', YEAR]].rename(columns={YEAR: 'imports_thousand_usd'}),
    # your code here: on the code column, outer, indicator, validate one to one
).rename(columns={'Código': 'country_code', 'País': 'country_name'})

print(trade['_merge'].value_counts())
trade = trade.drop(columns='_merge')
print('Merged:', trade.shape)

In [ ]:
def partner_profile(row):
    """Describe Angola's 2025 relationship with one partner.

    Reads both flows, so it has to run on the row rather than on a column.
    """
    sold = row['exports_thousand_usd']
    bought = row['imports_thousand_usd']

    if pd.isna(sold) or pd.isna(bought):
        return 'Incomplete'
    if sold + bought < 1000:
        return 'Negligible'
    # your code here: 'Angola mainly sells' when exports more than double
    # imports, 'Angola mainly buys' when the reverse, otherwise 'Two way'
    return


trade['balance_thousand_usd'] = (trade['exports_thousand_usd'].fillna(0)
                                 - trade['imports_thousand_usd'].fillna(0))
trade['profile'] = trade.apply(partner_profile, axis=1)

print(trade['profile'].value_counts())

In [ ]:
print('Largest surpluses:')
print(trade.nlargest(5, 'balance_thousand_usd')[
    ['country_name', 'exports_thousand_usd', 'imports_thousand_usd', 'profile']
].to_string(index=False))
print()
print('Largest deficits:')
print(trade.nsmallest(5, 'balance_thousand_usd')[
    ['country_name', 'exports_thousand_usd', 'imports_thousand_usd', 'profile']
].to_string(index=False))

**Questions:**

- Did every partner match? What does `validate='one_to_one'` promise?
- Which partners show the largest surplus and deficit? Does that fit what you
  know about Angola?
- The `Negligible` threshold is a judgement. What happens without it?
- What does `fillna(0)` assume about a missing flow?

**PT:** Todos os parceiros corresponderam? Que parceiros tem maior excedente e
defice? O limiar `Negligible` e um julgamento: o que acontece sem ele?

---

## Task 9: A hierarchy that will double count if you let it

The economic categories workbook (CGCE) codes a tree in the length of the code:
`1` is a section, `11` a group inside it, `111` a subgroup inside that. Summing
the column adds every level together.

A one line function applied to the code column exposes the structure, and the
published `Total Geral` is the check.

**PT:** O ficheiro de Grandes Categorias Economicas codifica uma arvore no
comprimento do codigo: `1` seccao, `11` grupo, `111` subgrupo. Somar a coluna
soma todos os niveis. O `Total Geral` publicado serve de verificacao.

In [ ]:
CGCE_FILE = 'Comercio Externo de Bens por Grandes Categorias Económicas.xlsx'
cgce = load_trade_sheet(CGCE_FILE, 'Export Cat. Económica (USD)', 'Descrição')
cgce['CGCE'] = cgce['CGCE'].astype('string')

published_total = cgce.loc[cgce['Descrição'] == 'Total Geral', YEAR].iloc[0]
cgce = cgce[cgce['CGCE'].notna()].copy()

print('Published Total Geral:', f'{published_total:,.0f}')
print('Category rows:', len(cgce))
cgce[['CGCE', 'Descrição', YEAR]].head()

In [ ]:
def cgce_level(code):
    """Depth in the CGCE tree: 1 section, 2 group, 3 subgroup."""
    # your code here: 0 when missing, otherwise the length of the code
    return


cgce['level'] = cgce['CGCE'].apply(cgce_level)
print(cgce['level'].value_counts().sort_index())

In [ ]:
naive =   # your code here: sum every row
sections_only =   # your code here: sum only the level 1 rows

print(f'Published total:        {published_total:15,.0f}')
print(f'Sum of every row:       {naive:15,.0f}  <- {naive / published_total:.2f}x')
print(f'Sum of level 1 only:    {sections_only:15,.0f}')
print()
print('Level 1 matches the published total:',
      bool(abs(sections_only - published_total) < 1))

**Questions:**

- How many rows sit at each level of the tree?
- Sum every row and compare to the published `Total Geral`. What is the ratio,
  and why is it exactly that?
- Which rows should you sum to reproduce the published figure?
- Nothing raised an error here. What would have caught this if you had not
  checked?

**PT:** Quantas linhas em cada nivel? Some todas as linhas e compare com o
`Total Geral`: qual e a razao e porque e exatamente essa? Que linhas deve somar?
Nada deu erro: o que teria detetado isto?

---

## Task 10: Stack the two flows

Merging combines columns, appending combines rows. Exports and imports have the
same shape, so a long format with a `flow` column is often easier to chart and
group than two wide columns.

**PT:** Juntar combina colunas, empilhar combina linhas. Exportacoes e
importacoes tem a mesma forma, por isso o formato longo com uma coluna `flow` e
mais facil de usar.

In [ ]:
long_exports = exports[['Código', 'País', YEAR]].assign(flow='Export')
long_imports = imports[['Código', 'País', YEAR]].assign(flow='Import')

flows =   # your code here: concat the two, ignore_index=True
flows = flows.rename(columns={'Código': 'country_code', 'País': 'country_name',
                              YEAR: 'value_thousand_usd'})

print('Stacked:', flows.shape)
print(flows['flow'].value_counts())
print()
print(flows.groupby('flow')['value_thousand_usd'].sum().round(0))

**Questions:**

- How many rows does the stacked frame have?
- Why add the `flow` column before the concat rather than after?
- Which flow is larger in 2025?

**PT:** Quantas linhas tem a tabela empilhada? Porque adicionar a coluna `flow`
antes do concat? Qual fluxo e maior em 2025?

---

## Task 11: Save

**PT:** Gravar os resultados.

In [ ]:
os.makedirs(DATA_PROC_DIR, exist_ok=True)

survey_out = os.path.join(DATA_PROC_DIR, 'angola_iea_2025q4_features.csv')
trade_out = os.path.join(DATA_PROC_DIR, 'angola_trade_partners.csv')
flows_out = os.path.join(DATA_PROC_DIR, 'angola_trade_flows.csv')

# your code here: write the three frames with index=False

print('survey:', df.shape, '| trade:', trade.shape, '| flows:', flows.shape)

**Questions:**

- Which three files did you write, and what is each for?
- Which columns did the survey gain, and which of them came from a function?

**PT:** Que tres ficheiros gravou e para que serve cada um? Que colunas ganhou o
inquerito e quais vieram de uma funcao?